In [ ]:
# CELL 0 Install dependencies & mount Google Drive - setup
!pip install -q torch torchvision
!pip install -q pandas numpy pyyaml matplotlib opencv-python-headless

import os
if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5
    !pip install -q -r /content/yolov5/requirements.txt

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 1 Imports
import os
import sys
import json
import shutil
import random
import subprocess
import re
import numpy as np
import pandas as pd
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

from IPython import display
from IPython.display import display, Image

sys.path.insert(0, '/content/yolov5')

print(f'torch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# CELL 2 Constants / Config
HOME = os.getcwd(); print(HOME)

# Experiment selector
# Pick ONE:
#   'EXP1_ALL' (all datasets combined)
#   'EXP2_OCCLUSION'
#   'EXP3_CLUSTER'
#   'EXP4_ILLUMINATION'
#   'EXP5_SCALE'
EXPERIMENT = 'EXP5_SCALE'

# Unified split paths (YOLO format, pre-created on Drive)
# The unified split is created ONCE by the split-creation notebook and stored at this path
UNIFIED_ROOT_DRIVE = f'/content/drive/MyDrive/Dataset/{EXPERIMENT}/YOLO_BOX'

# Run config
RUN_STAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME  = f'{EXPERIMENT}_{RUN_STAMP}'
RUN_DIR   = f'/content/drive/MyDrive/Oranges/YOLOv5_Runs/{RUN_NAME}'

IMAGE_EXTS = {'.jpg', '.jpeg'}

CONFIG = {
    'epochs'    : 100,
    'batch_size': 16,
    'image_size': 640,
    'save_path' : f'{HOME}/results',   # local Colab results dir
}

os.makedirs(RUN_DIR,           exist_ok=True)
os.makedirs(CONFIG['save_path'], exist_ok=True)

# Save config snapshot to Drive immediately
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump({'experiment': EXPERIMENT, 'run_stamp': RUN_STAMP, **CONFIG}, f, indent=2)

print(f'Experiment     : {EXPERIMENT}')
print(f'Unified split  : {UNIFIED_ROOT_DRIVE}')
print(f'Run dir (Drive): {RUN_DIR}')


In [ ]:
# CELL 2b Copy dataset to local storage

LOCAL_ROOT = f'/content/unified_split/{EXPERIMENT}/YOLO_BOX'

if not os.path.exists(LOCAL_ROOT):
    print('Copying unified split to local storage (one-time cost)...')
    shutil.copytree(UNIFIED_ROOT_DRIVE, LOCAL_ROOT)
    print('Done')
else:
    print('Local copy already exists - skipping')

TRAIN_DIR = os.path.join(LOCAL_ROOT, 'train')
VALID_DIR = os.path.join(LOCAL_ROOT, 'valid')
TEST_DIR  = os.path.join(LOCAL_ROOT, 'test')
YAML_PATH = os.path.join(LOCAL_ROOT, 'data.yaml')

# Patch data.yaml to point at local paths (YOLOv5 train.py reads this)
with open(YAML_PATH, 'r') as f:
    yaml_data = yaml.safe_load(f)

yaml_data['path']  = LOCAL_ROOT
yaml_data['train'] = 'train/images'
yaml_data['val']   = 'valid/images'
yaml_data['test']  = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False)

print(f'Train dir : {TRAIN_DIR}')
print(f'Valid dir : {VALID_DIR}')
print(f'Test dir  : {TEST_DIR}')
print(f'YAML      : {YAML_PATH}')


In [ ]:
# CELL 3 Verify unified split
def count_split(split_dir):
    img_dir = Path(split_dir) / 'images'
    lbl_dir = Path(split_dir) / 'labels'
    imgs = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    oranges = sum(
        sum(1 for line in open(lbl).readlines() if line.strip())
        for lbl in lbl_dir.glob('*.txt')
    )
    return imgs, oranges

print(f'{"Split":<8} {"Images":>8} {"Oranges":>10}')
print('-' * 30)
for name, d in [('train', TRAIN_DIR), ('valid', VALID_DIR), ('test', TEST_DIR)]:
    imgs, oranges = count_split(d)
    print(f'{name:<8} {imgs:>8} {oranges:>10}')


In [ ]:
# CELL 4 YOLOv5 Training

def val_input_v5(model_size):
    valid_sizes = {'n', 's', 'm', 'l', 'x'}
    if model_size not in valid_sizes:
        raise ValueError(f'model_size must be one of {valid_sizes}')

def run_model(model_size, dataset_choice, trace=True):

    val_input_v5(model_size)

    model_name = f'yolov5{model_size}.pt'
    run_name = f'yolov5{model_size}_{dataset_choice}_checkpoint'
    run_dir = os.path.join(CONFIG['save_path'], run_name)

    train_cmd = (
        f"python /content/yolov5/train.py "
        f"--img {CONFIG['image_size']} "
        f"--batch {CONFIG['batch_size']} "
        f"--epochs {CONFIG['epochs']} "
        f"--data {YAML_PATH} "
        f"--weights {model_name} "
        f"--project {CONFIG['save_path']} "
        f"--name {run_name} "
        f"--exist-ok "
        f"--save-period 10 "
        f"--cache ram"
    )

    proc = subprocess.Popen(
        train_cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'[ERROR] Training failed (return code {proc.returncode})')

    best_path = os.path.join(run_dir, 'weights', 'best.pt')

    if not os.path.exists(best_path):
        print(f'[ERROR] best.pt not found: {best_path}')
    elif trace:
        for fname in ['results.png', 'confusion_matrix.png']:
            fpath = os.path.join(run_dir, fname)
            if os.path.exists(fpath):
                display(Image(filename=fpath, width=700))

    return run_dir, best_path


In [ ]:
# CELL 5 Evaluation / Metrics

def ground_truth_count(label_path):
    if not os.path.exists(label_path):
        return 0
    with open(label_path) as f:
        return sum(1 for line in f if line.strip())

def eval_model(best_path, conf=0.25):

    if not os.path.exists(best_path):
        print(f'[ERROR] best.pt not found: {best_path}')
        return {k: float('nan') for k in
                ['box_precision','box_recall','box_f1','box_map50','box_map','mae','rmse']}

    val_cmd = (
        f"python /content/yolov5/val.py "
        f"--weights {best_path} "
        f"--data {YAML_PATH} "
        f"--img {CONFIG['image_size']} "
        f"--task test "
        f"--conf-thres {conf} "
        f"--project {CONFIG['save_path']} "
        f"--name eval_output "
        f"--exist-ok"
    )
    result = subprocess.run(val_cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    output = result.stdout

    ansi = re.compile(r'(\x9B|\x1B\[)[0-?]*[ -/]*[@-~]')
    clean = '\n'.join(ansi.sub('', l).replace('\r','') for l in output.split('\n'))

    pattern = re.compile(
        r'\ball\s+\d+\s+\d+\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)'
    )
    match = pattern.search(output) or pattern.search(clean)
    if match:
        box_p     = float(match.group(1))
        box_r     = float(match.group(2))
        box_map50 = float(match.group(3))
        box_map   = float(match.group(4))
        print(f'Parsed - P:{box_p:.3f}  R:{box_r:.3f}  mAP50:{box_map50:.3f}  mAP50-95:{box_map:.3f}')
    else:
        print('[WARNING] Could not parse val.py output.')
        print(output[-2000:])
        box_p = box_r = box_map50 = box_map = float('nan')

    box_f1 = (2*box_p*box_r/(box_p+box_r)) if (box_p and box_r and (box_p+box_r)>0) else 0.0

    # Count-based MAE / RMSE via detect.py
    test_img_dir   = Path(TEST_DIR) / 'images'
    test_label_dir = Path(TEST_DIR) / 'labels'
    image_paths    = sorted([p for p in test_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

    detect_out_dir = os.path.join(CONFIG['save_path'], 'detect_tmp_eval')
    pred_label_dir = Path(detect_out_dir) / 'preds' / 'labels'
    if pred_label_dir.exists():
        shutil.rmtree(str(pred_label_dir))

    detect_cmd = (
        f"python /content/yolov5/detect.py "
        f"--weights {best_path} "
        f"--source {str(test_img_dir)} "
        f"--conf-thres {conf} "
        f"--img {CONFIG['image_size']} "
        f"--save-txt --save-conf "
        f"--project {detect_out_dir} "
        f"--name preds --exist-ok"
    )
    os.system(detect_cmd)

    errors = []
    for img_path in image_paths:
        gt  = ground_truth_count(str(test_label_dir / f'{img_path.stem}.txt'))
        pred_lbl = pred_label_dir / f'{img_path.stem}.txt'
        pred = ground_truth_count(str(pred_lbl)) if pred_lbl.exists() else 0
        errors.append(pred - gt)

    errors = np.array(errors, dtype=float)
    return {
        'box_precision': box_p,
        'box_recall' : box_r,
        'box_f1' : box_f1,
        'box_map50' : box_map50,
        'box_map' : box_map,
        'mae' : float(np.mean(np.abs(errors))),
        'rmse' : float(np.sqrt(np.mean(errors**2))),
    }


In [ ]:
# CELL 6 Confidence threshold sweep

def search_conf_threshold(best_path, thresholds=None, model_size='?'):

    if thresholds is None:
        thresholds = [round(x, 2) for x in np.arange(0.1, 0.95, 0.05)]

    val_img_dir   = Path(VALID_DIR) / 'images'
    val_label_dir = Path(VALID_DIR) / 'labels'
    image_paths   = sorted([p for p in val_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

    detect_out_dir = os.path.join(CONFIG['save_path'], 'conf_search')
    pred_label_dir = Path(detect_out_dir) / 'preds' / 'labels'
    results = []

    for conf in thresholds:
        if pred_label_dir.exists():
            shutil.rmtree(str(pred_label_dir))

        detect_cmd = (
            f"python /content/yolov5/detect.py "
            f"--weights {best_path} "
            f"--source {str(val_img_dir)} "
            f"--conf-thres {conf} "
            f"--img {CONFIG['image_size']} "
            f"--save-txt "
            f"--project {detect_out_dir} "
            f"--name preds --exist-ok"
        )
        os.system(detect_cmd)

        errors = []
        for img_path in image_paths:
            gt  = ground_truth_count(str(val_label_dir / f'{img_path.stem}.txt'))
            pred_lbl = pred_label_dir / f'{img_path.stem}.txt'
            pred = ground_truth_count(str(pred_lbl)) if pred_lbl.exists() else 0
            errors.append(pred - gt)

        arr  = np.array(errors, dtype=float)
        mae  = float(np.mean(np.abs(arr)))
        rmse = float(np.sqrt(np.mean(arr**2)))
        results.append({'conf': conf, 'mae': mae, 'rmse': rmse})
        print(f'conf={conf:.2f}  MAE={mae:.3f}  RMSE={rmse:.3f}')

    df       = pd.DataFrame(results)
    best_row = df.loc[df['mae'].idxmin()]
    best_conf = float(best_row['conf'])
    print(f'\nBest threshold (YOLOv5{model_size}): {best_conf}  (MAE={best_row["mae"]:.3f})')

    # Save sweep plot
    plt.figure(figsize=(7, 4))
    plt.plot(df['conf'], df['mae'],  marker='o', label='MAE')
    plt.plot(df['conf'], df['rmse'], marker='s', label='RMSE')
    plt.axvline(best_conf, color='red', linestyle='--', label=f'best={best_conf}')
    plt.xlabel('Confidence threshold'); plt.ylabel('Error')
    plt.title(f'Conf Sweep - YOLOv5{model_size}')
    plt.legend(); plt.tight_layout()
    sweep_png = os.path.join(CONFIG['save_path'], f'conf_sweep_{model_size}.png')
    plt.savefig(sweep_png, dpi=120)
    plt.show()

    return df, best_conf, sweep_png


In [ ]:
# CELL 7 Save run outputs to Drive

def save_model_run_to_drive(model_size, run_dir, best_path,
                             metrics, sweep_df, sweep_png):

    size_drive_dir = os.path.join(RUN_DIR, model_size)
    os.makedirs(size_drive_dir, exist_ok=True)

    # Weights
    if os.path.exists(best_path):
        shutil.copy2(best_path, os.path.join(size_drive_dir, 'best.pt'))

    # Training curve plots
    for fname in ['results.png', 'confusion_matrix.png']:
        src = os.path.join(run_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(size_drive_dir, fname))

    # Metrics JSON
    with open(os.path.join(size_drive_dir, 'metrics.json'), 'w') as f:
        json.dump({k: (float(v) if isinstance(v, (float, int)) else v)
                   for k, v in metrics.items()}, f, indent=2)

    # Confidence sweep
    sweep_df.to_csv(os.path.join(size_drive_dir, 'conf_sweep.csv'), index=False)
    if os.path.exists(sweep_png):
        shutil.copy2(sweep_png, os.path.join(size_drive_dir, 'conf_sweep.png'))

    print(f'  Saved YOLOv5{model_size} outputs to {size_drive_dir}')


In [ ]:
# CELL 8 Orchestrator: train all 5 sizes + save everything

def all_model_runs():
    rows = []

    for ms in ['n', 's', 'm', 'l', 'x']:
        print(f'\n--- training yolov5{ms} ---')

        start = datetime.now()
        run_dir, best_path = run_model(ms, EXPERIMENT)
        end   = datetime.now()

        sweep_df, best_conf, sweep_png = search_conf_threshold(
            best_path, model_size=ms
        )
        metrics = eval_model(best_path=best_path, conf=best_conf)

        save_model_run_to_drive(
            model_size=ms,
            run_dir=run_dir,
            best_path=best_path,
            metrics=metrics,
            sweep_df=sweep_df,
            sweep_png=sweep_png,
        )

        rows.append({
            'experiment' : EXPERIMENT,
            'model'      : 'YOLOv5',
            'model_size' : ms,
            'task'       : 'box',
            'run_dir'    : run_dir,
            'best_path'  : best_path,
            'conf'       : best_conf,
            'train_time' : str(end - start),
            **metrics,
        })

    results_df = pd.DataFrame(rows)

    # Save summary CSV locally + to Drive
    local_csv  = os.path.join(CONFIG['save_path'], 'summary.csv')
    drive_csv  = os.path.join(RUN_DIR, 'summary.csv')
    results_df.to_csv(local_csv,  index=False)
    results_df.to_csv(drive_csv,  index=False)
    print(f'\nSummary saved to Drive: {drive_csv}')

    return results_df

torch.cuda.empty_cache()
results_df = all_model_runs()
results_df


In [ ]:
# CELL 9 Visualise GT vs predictions on test set
import random as _random

def visualize_predictions(best_path, best_conf, model_size, n_samples=10):
    test_img_dir   = Path(TEST_DIR) / 'images'
    test_label_dir = Path(TEST_DIR) / 'labels'
    image_paths    = sorted([p for p in test_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

    _random.seed(42)
    sample_paths = _random.sample(image_paths, min(n_samples, len(image_paths)))

    detect_out_dir = os.path.join(CONFIG['save_path'], 'viz_detect')
    detect_cmd = (
        f"python /content/yolov5/detect.py "
        f"--weights {best_path} "
        f"--source {str(test_img_dir)} "
        f"--conf-thres {best_conf} "
        f"--img {CONFIG['image_size']} "
        f"--save-txt "
        f"--project {detect_out_dir} "
        f"--name preds --exist-ok --nosave"
    )
    os.system(detect_cmd)
    pred_label_dir = Path(detect_out_dir) / 'preds' / 'labels'

    fig, axes = plt.subplots(len(sample_paths), 2,
                             figsize=(14, 6 * len(sample_paths)))

    for row, img_path in enumerate(sample_paths):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w    = img_rgb.shape[:2]

        gt_overlay = img_rgb.copy()
        gt_count   = 0
        gt_lbl = test_label_dir / f'{img_path.stem}.txt'
        if gt_lbl.exists():
            for line in open(gt_lbl):
                parts = line.strip().split()
                if len(parts) != 5: continue
                _, cx, cy, bw, bh = map(float, parts)
                x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
                x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
                cv2.rectangle(gt_overlay,(x1,y1),(x2,y2),(0,255,0),2)
                gt_count += 1

        pred_overlay = img_rgb.copy()
        pred_count   = 0
        pred_lbl = pred_label_dir / f'{img_path.stem}.txt'
        if pred_lbl.exists():
            for line in open(pred_lbl):
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, bw, bh = map(float, parts[:5])
                x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
                x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
                cv2.rectangle(pred_overlay,(x1,y1),(x2,y2),(255,0,0),2)
                pred_count += 1

        axes[row,0].imshow(gt_overlay)
        axes[row,0].set_title(f'Ground Truth (count: {gt_count})', fontsize=13, fontweight='bold')
        axes[row,0].axis('off')
        axes[row,1].imshow(pred_overlay)
        axes[row,1].set_title(f'Predicted (count: {pred_count}, conf≥{best_conf:.2f})', fontsize=13, fontweight='bold')
        axes[row,1].axis('off')

    plt.suptitle(f'YOLOv5{model_size} - GT vs Prediction [{EXPERIMENT}]',
                 fontsize=15, fontweight='bold', y=1.0)
    plt.tight_layout()

    # Save viz to Drive
    viz_path = os.path.join(RUN_DIR, model_size, 'test_predictions.png')
    plt.savefig(viz_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {viz_path}')


# Run viz for best model (x) - change model_size to whichever you want
best_row = results_df.loc[results_df['box_map50'].idxmax()]
visualize_predictions(
    best_path  = best_row['best_path'],
    best_conf  = best_row['conf'],
    model_size = best_row['model_size'],
)


In [ ]:
# CELL 10 Print image / orange counts per split
from pathlib import Path

for split_name, split_dir in [('train', TRAIN_DIR), ('valid', VALID_DIR), ('test', TEST_DIR)]:
    imgs = len([p for p in (Path(split_dir)/'images').iterdir()
                if p.suffix.lower() in IMAGE_EXTS])
    oranges = sum(
        sum(1 for line in open(lbl) if line.strip())
        for lbl in (Path(split_dir)/'labels').glob('*.txt')
    )
    print(f'{split_name}: {imgs} images  {oranges} oranges')
